In [1]:
import pandas as pd
import numpy as np
import json
import re
from ast import literal_eval
import importlib
from dotenv import load_dotenv
import os
os.chdir('..')


In [2]:
# Load environment variables from .env file
load_dotenv()

True

In [3]:
# Set GEMINI_API_KEY environment variable with your API key
os.environ['GEMINI_API_KEY'] = os.getenv('GEMINI_API_KEY')

In [4]:
import base64
from google import genai
from google.genai import types
from copy import deepcopy

In [5]:
def generate(prompt):
	client = genai.Client(
			api_key=os.environ.get("GEMINI_API_KEY"),
		)

	model = "gemini-2.5-flash"
	contents = [
		types.Content(
			role="user",
			parts=[
				types.Part.from_text(
					text=prompt
				),
			],
		),
	]
	generate_content_config = types.GenerateContentConfig(
		temperature=0.75,
		top_p=0.9,
		top_k=40,
		max_output_tokens=8192,
		thinking_config=types.ThinkingConfig(thinking_budget=-1), # Dynamic thinking = -1, no thinking = 0
		response_mime_type="application/json",
		system_instruction=[
			types.Part.from_text(
				text="""You are an expert in Natural Language Processing. You are also linguist with an expertise in Indonesian and English."""
			),
		],
	)
		
	response = client.models.generate_content(
		model=model, contents=contents, config=generate_content_config
	)
	# print(response.text)
	return response

from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

In [6]:
dataset_folder = 'mvp_aos'
dataset_type = 'hotel_reviews'
lang = 'indo'
split = 'train'
dataset_path = f'dataset/{dataset_type}/{lang}/{dataset_folder}/{split}.json'
with open(dataset_path, 'r') as f:
	dataset = json.load(f)


In [7]:
dataset_unique = []
for instance in dataset:
	if instance['element_order'] == 'aos':
		dataset_unique.append(deepcopy(instance))

In [8]:
dataset_single_triplet = []
for instance in dataset_unique:
	triplets = parse_absa_string(instance['target'])
	for triplet in triplets:
		new_instance = deepcopy(instance)
		new_instance['target'] = f"[A] {triplet.get('A', '')} [O] {triplet.get('O', '')} [S] {triplet.get('S', '')}"
		dataset_single_triplet.append(new_instance)

In [9]:
df_dataset = pd.DataFrame(dataset_unique)
df_dataset

,sentence_id,instance_id,input,target,element_order,task_elements
0,0,0,kamar saya ada kendala di ac tidak berfungsi o...,[A] ac [O] tidak berfungsi optimal [S] negativ...,aos,aos
1,1,5,tempatnya bagus . kolam renangnya bersih . [A]...,[A] tempatnya [O] bagus [S] positive [SSEP] [A...,aos,aos
2,2,10,"oke banget , tetapi ac nya tidak bisa diatur s...",[A] ac nya [O] tidak bisa diatur suhu nya [S] ...,aos,aos
3,3,15,keren . nyaman semuanya . [A] [O] [S],[A] semuanya [O] nyaman [S] positive [SSEP] [A...,aos,aos
4,4,20,"tidak dapat snack . setelah di keluhan , baru ...",[A] snack [O] tidak dapat [S] negative,aos,aos
...,...,...,...,...,...,...
2477,2495,12475,wifi kurang joss . [A] [O] [S],[A] wifi [O] kurang joss [S] negative,aos,aos
2478,2496,12480,"kamar cukup bersih , hanya sempit , . [A] [O] [S]",[A] kamar [O] cukup bersih [S] positive [SSEP]...,aos,aos
2479,2497,12485,"nyaman , bersih , dan pelayananya sangat ramah...",[A] pelayananya [O] sangat ramah [S] positive ...,aos,aos
2480,2498,12490,sangat kecewa dengan kamar dan pelayanan stafn...,[A] kamar [O] sangat kecewa [S] negative [SSEP...,aos,aos


In [10]:
check_list = df_dataset['sentence_id'].to_list()
# Check if check_list is ordered from smallest to largest (missing indexes is allowed, just make sure the order is correct)
is_ordered = all(earlier <= later for earlier, later in zip(check_list, check_list[1:]))
print(f"Is the sentence_id list ordered? {is_ordered}")

Is the sentence_id list ordered? True


In [11]:
lang_target = 'mad'
with open(f'notebooks/prompt_translate/translate-triplet-{lang_target}.txt', 'r') as f:
	prompt_template = f.read()
print(prompt_template)

### Instruction
You will be given input-output pairs of Aspect Sentiment Triplet Extraction.
Given an Indonesian text and triplets consist of aspect term, opinion term, and sentiment polarity, translate all of them to Enjek-Iye level of Madurese. Enjek-Iye level Madurese is the informal variety of Madurese used in daily conversations between friends or people of the same age.
The order of the triplet is (aspect term, opinion term, sentiment polarity).
Below is the definition of each element in the triplet:
- The aspect term refers to a specific feature, attribute, or aspect of a product or service on which a user can express an opinion. Explicit aspect terms appear explicitly as a substring of the given text. The aspect term might be “NULL” for the implicit aspect.
- The sentiment polarity refers to the degree of positivity, negativity or neutrality expressed in the opinion towards a particular aspect or feature of a product or service, and the available polarities include: “positive”,

### Individual API request

In [ ]:
from tqdm import tqdm
from time import sleep
from ast import literal_eval

outputs = {}
outputs_text = {}
for idx, row in tqdm(df_dataset.iterrows(), desc="Translating triplets", total=df_dataset.shape[0]):
	input_text = row['input'].replace('[A] [O] [S]', '').strip()
	triplets = parse_absa_string(row['target'])
	target_text = []
	for triplet in triplets:
		target_text.append(f"({triplet.get('A', 'err_empty')}, {triplet.get('O', 'err_empty')}, {triplet.get('S', 'err_empty')})")
	target_text = '[' + ', '.join(target_text) + ']'
	prompt = prompt_template.replace('{text-input}', input_text).replace('{input-triplets}', target_text)
	while True:
		try:
			output = generate(prompt)
			outputs[idx] = output
			outputs_text[idx] = literal_eval(output.text)
			outputs_text[idx]['text'] = input_text
			outputs_text[idx]['triplets'] = target_text
			break
		except Exception as e:
			print(f"Error: {e}")
			sleep(3.0)  # Wait for 3 seconds before retrying
			continue
	
	# Write to json file after each successful generation
	os.makedirs(f'temp/translation_output/eng/{dataset_folder}', exist_ok=True)
	with open(f'temp/translation_output/eng/{dataset_folder}/{os.path.basename(dataset_path)}', 'w') as f:
		json.dump(outputs_text, f, indent=4, ensure_ascii=False)
	
	if idx == 10:
		break

Translating triplets:   0%|          | 0/2482 [00:03<?, ?it/s]


KeyboardInterrupt: 

### Batch API request

In [12]:
targetlang2langname = {
	'indo': 'Indonesian',
    'sunda': 'Sundanese',
    'jav': 'Javanese',
    'min': 'Minangkabau',
    'mad': 'Madurese',
}

In [13]:
inline_requests = []
temp_for_jsonl = []
for idx, row in df_dataset.iterrows():
	input_text = row['input'].replace('[A] [O] [S]', '').strip()
	triplets = parse_absa_string(row['target'])
	target_text = []
	for triplet in triplets:
		target_text.append(f"({triplet.get('A', 'err_empty')}, {triplet.get('O', 'err_empty')}, {triplet.get('S', 'err_empty')})")
	target_text = '[' + ', '.join(target_text) + ']'
	prompt = prompt_template.replace('{text-input}', input_text).replace('{input-triplets}', target_text)
	
	inline_request = {
		"contents": [
			{
				"role": "user",
				"parts": [
					{
						"text": prompt
					}
				]
			}
		],
		"generation_config": {
			"temperature": 0.75,
			"top_p": 0.9,
			"top_k": 40,
			"max_output_tokens": 8192,
			"thinking_config": {
				"thinking_budget": -1
			},
			"response_mime_type": "application/json",
		},
		"system_instruction": {
			"parts": [
				{
					'text': f"You are an expert in Natural Language Processing. You are also linguist with an expertise in Indonesian and {targetlang2langname[lang_target]}."
				}
			]
		}
	}
	inline_requests.append(inline_request)

	temp_for_jsonl.append({
		'key': f'request-{row["sentence_id"]}',
		'request': inline_request
	})
	# if idx == 4:
	# 	break

In [14]:
len(inline_requests), len(temp_for_jsonl)

(2482, 2482)

In [15]:
# print(inline_requests[-10]['contents'][0]['parts'][0]['text'])
print(temp_for_jsonl[-1])

{'key': 'request-2499', 'request': {'contents': [{'role': 'user', 'parts': [{'text': '### Instruction\nYou will be given input-output pairs of Aspect Sentiment Triplet Extraction.\nGiven an Indonesian text and triplets consist of aspect term, opinion term, and sentiment polarity, translate all of them to Enjek-Iye level of Madurese. Enjek-Iye level Madurese is the informal variety of Madurese used in daily conversations between friends or people of the same age.\nThe order of the triplet is (aspect term, opinion term, sentiment polarity).\nBelow is the definition of each element in the triplet:\n- The aspect term refers to a specific feature, attribute, or aspect of a product or service on which a user can express an opinion. Explicit aspect terms appear explicitly as a substring of the given text. The aspect term might be “NULL” for the implicit aspect.\n- The sentiment polarity refers to the degree of positivity, negativity or neutrality expressed in the opinion towards a particular 

In [16]:
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

In [17]:
# Create a jsonl file for inline requests
os.makedirs(f'temp/translation_output/{lang_target}/{dataset_folder}', exist_ok=True)
with open(f'temp/translation_output/{lang_target}/{dataset_folder}/inline_requests_{lang_target}_{split}.jsonl', 'w') as f:
	for item in temp_for_jsonl:
		f.write(json.dumps(item, ensure_ascii=False) + '\n')

In [18]:
# Upload the file to the File API
uploaded_file = client.files.upload(
    file=f'temp/translation_output/{lang_target}/{dataset_folder}/inline_requests_{lang_target}_{split}.jsonl',
    config=types.UploadFileConfig(display_name=f'inline_requests_{lang_target}_{split}', mime_type='jsonl')
)

print(f"Uploaded file: {uploaded_file.name}")

Uploaded file: files/ix88t08idmt2


In [21]:
from google import genai

# Assumes `uploaded_file` is the file object from the previous step
file_batch_job = client.batches.create(
    model="gemini-2.5-flash",
    src=uploaded_file.name,
    config={
        'display_name': f"file-upload-job-translation-{lang_target}-{split}-fixed",
    },
)

print(f"Created batch job: {file_batch_job.name}")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/usage?tab=rate-limit. ', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}]}}

In [20]:
# List all batches
batches = client.batches.list()
for batch in batches:
    print(f"Batch ID: {batch.name}, State: {batch.state.name}, Display Name: {batch.display_name}")

Batch ID: batches/j8rg2se3a2hzp4ziaspq8n81f1l6687hdv0d, State: JOB_STATE_PENDING, Display Name: file-upload-job-translation-min-train
Batch ID: batches/94ipo9kv0sh5r0923lpiqib0onz3wvfjdqs3, State: JOB_STATE_SUCCEEDED, Display Name: file-upload-job-translation-mad-train
Batch ID: batches/aaodv2jutswgb1vrjs8sg2iikhluo5udboxe, State: JOB_STATE_SUCCEEDED, Display Name: None
Batch ID: batches/dfkyvx1ka9aj8xhh4zt6w47xgptxvbki75as, State: JOB_STATE_SUCCEEDED, Display Name: None
Batch ID: batches/40ogifgcu82d4tx1vk9gmg97mcoqlq3z7l94, State: JOB_STATE_SUCCEEDED, Display Name: None
Batch ID: batches/filprkchw3b7wpcl38haxk9p6fe9boj4dls2, State: JOB_STATE_SUCCEEDED, Display Name: hoasa-augment-job
Batch ID: batches/9neirboro5zb7gpqa4u3o99tztr6kj0kt67s, State: JOB_STATE_SUCCEEDED, Display Name: hoasa-augment-job
Batch ID: batches/lpxh4xaknvxckykmrk9hjbazv87anyyklal4, State: JOB_STATE_SUCCEEDED, Display Name: file-upload-job-translation-jav-train-fixed
Batch ID: batches/9udm06rnexogkfmqcxmav7oblt1g6

In [ ]:
from time import sleep

# Use the name of the job you want to check
# e.g., inline_batch_job.name from the previous step
job_name = "batches/9udm06rnexogkfmqcxmav7oblt1g6lm467z3"  # (e.g. 'batches/your-batch-id')
batch_job = client.batches.get(name=job_name)

completed_states = set([
    'JOB_STATE_SUCCEEDED',
    'JOB_STATE_FAILED',
    'JOB_STATE_CANCELLED',
    'JOB_STATE_EXPIRED',
])

print(f"Polling status for job: {job_name}")
batch_job = client.batches.get(name=job_name) # Initial get
while batch_job.state.name not in completed_states:
  print(f"Current state: {batch_job.state.name}")
  sleep(30) # Wait for 30 seconds before polling again
  batch_job = client.batches.get(name=job_name)

print(f"Job finished with state: {batch_job.state.name}")
if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"Error: {batch_job.error}")

Polling status for job: batches/9udm06rnexogkfmqcxmav7oblt1g6lm467z3
Job finished with state: JOB_STATE_SUCCEEDED


In [ ]:
from tqdm import tqdm

In [ ]:
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

# Use the name of the job you want to check
# e.g., inline_batch_job.name from the previous step
job_name = "batches/9udm06rnexogkfmqcxmav7oblt1g6lm467z3"
batch_job = client.batches.get(name=job_name)

outputs = {}
outputs_text = {}

if batch_job.state.name == 'JOB_STATE_SUCCEEDED':

	# If batch job was created with a file
	if batch_job.dest and batch_job.dest.file_name:
		# Results are in a file
		result_file_name = batch_job.dest.file_name
		print(f"Results are in file: {result_file_name}")

		print("Downloading result file content...")
		file_content = client.files.download(file=result_file_name)
		# Process file_content (bytes) as needed
		lines = file_content.decode('utf-8').splitlines()
		for i, line in tqdm(enumerate(lines)):
			result = json.loads(line)
			key = result['key'].split('-')[-1]

			# print(result)
			if 'response' in result:
				try:
					outputs_text[key] = literal_eval(result['response']['candidates'][0]['content']['parts'][0]['text'])
				except Exception as e:
					print(f"Error in Response {i+1}: {e}")
					outputs_text[key] = result['response']  # Fallback
					outputs_text[key] = {
						'translated_text': '',
						'translated_triplets': [],
					}
			elif 'error' in result:
				print(f"Error: {result['error']}")

	# If batch job was created with inline request
	# (for embeddings, use batch_job.dest.inlined_embed_content_responses)
	elif batch_job.dest and batch_job.dest.inlined_responses:
		# Results are inline
		print("Results are inline:")
		for i, inline_response in tqdm(enumerate(batch_job.dest.inlined_responses)):
			if inline_response.response:
				# Accessing response, structure may vary.
				try:
					outputs_text[i] = literal_eval(inline_response.response.text)
				except Exception as e:
					print(f"Error in Response {i+1}: {e}")
					outputs_text[i] = inline_response.response  # Fallback
					outputs_text[i] = {
						'translated_text': '',
						'translated_triplets': [],
					}
			elif inline_response.error:
				print(f"Error: {inline_response.error}")
	else:
		print("No results found (neither file nor inline).")
else:
	print(f"Job did not succeed. Final state: {batch_job.state.name}")
	if batch_job.error:
		print(f"Error: {batch_job.error}")

Results are in file: files/batch-9udm06rnexogkfmqcxmav7oblt1g6lm467z3


1000it [00:00, 15343.74it/s]

Error in Response 400: 'content'


In [ ]:
# for i, inline_response in tqdm(enumerate(batch_job.dest.inlined_responses)):
# 	if inline_response.response:
# 		# Accessing response, structure may vary.
# 		try:
# 			print(f"----------------- Response {i+1} -----------------")
# 			print(literal_eval(inline_response.response.text))
# 			print(inline_requests[i]['contents'][0]['parts'][0]['text'].split('### Inference:')[-1].strip())
# 		except Exception as e:
# 			print(f"Error in Response {i+1}: {e}")
# 			# outputs[i] = inline_response.response  # Fallback
# 			print(inline_response.response.text)
# 	elif inline_response.error:
# 		print(f"Error: {inline_response.error}")

In [ ]:
dataset_path

'dataset/hotel_reviews/indo/mvp_aos/test.json'

In [ ]:
save_path = f'temp/translation_output/{lang_target}/{dataset_folder}/{os.path.basename(dataset_path)}'

In [ ]:
save_path

'temp/translation_output/jav/mvp_aos/test.json'

In [ ]:
os.makedirs(f'temp/translation_output/{lang_target}/{dataset_folder}', exist_ok=True)
with open(save_path, 'w') as f:
	json.dump(outputs_text, f, indent=4, ensure_ascii=False)
print(f"Saved translated outputs to {save_path}")

Saved translated outputs to temp/translation_output/jav/mvp_aos/test.json


## Preprocess

In [ ]:
with open(save_path, 'r') as f:
	outputs_text = json.load(f)

In [ ]:
def add_space_around_punctuation(text):
    # Except for '-'
    # Ensure space before punctuation
    text = re.sub(r'(\S)([.,!?\(\)\"\';:+/]+)', r'\1 \2', text)
    # Ensure space after punctuation
    text = re.sub(r'([.,!?\(\)\"\';:+/]+)(\S)', r'\1 \2', text)
    # Replace multiple spaces with a single space
    text = re.sub(r'\s+', ' ', text)
    # Ensure punctuation sequences like '...' are split into spaced dots
    text = re.sub(r'([.]{2,})', lambda m: ' '.join(m.group(1)), text)
    return text.strip()

In [ ]:
len(outputs_text)

1000

In [ ]:
df_dataset

,sentence_id,instance_id,input,target,element_order,task_elements
0,3500,17500,pelayanan nya sangat ramah . [A] [O] [S],[A] pelayanan nya [O] sangat ramah [S] positive,aos,aos
1,3501,17505,sayang wifi tidak bagus harus keluar kamar . [...,[A] wifi [O] tidak bagus harus keluar kamar [S...,aos,aos
2,3502,17510,"tulisannya twin bed , tetapi yang ada kamarnya...",[A] kamarnya [O] beda [S] negative,aos,aos
3,3503,17515,"over all baik , hanya sja akan lebih memuaskan...",[A] over all [O] baik [S] positive [SSEP] [A] ...,aos,aos
4,3504,17520,fasilatas sesuia . [A] [O] [S],[A] fasilatas [O] sesuia [S] positive,aos,aos
...,...,...,...,...,...,...
995,4495,22475,"lumayan , harga murah banged . [A] [O] [S]",[A] harga [O] murah banged [S] positive [SSEP]...,aos,aos
996,4496,22480,buat lakilaki dan perempuan yang belum menikah...,[A] null [O] buat lakilaki dan perempuan yang ...,aos,aos
997,4497,22485,"kamar sangat nyaman dan bersih , sungguh menye...",[A] kamar [O] sangat nyaman [S] positive [SSEP...,aos,aos
998,4498,22490,"kamarnya luas , kasurnya empuk , kamar mandiny...",[A] kamarnya [O] luas [S] positive [SSEP] [A] ...,aos,aos


In [ ]:
def check_mismatches_triplet_format(outputs_text):
	mismatch_indexes = []
	mismatch_notes = {}
	for key, instance in outputs_text.items():
		translated_text = add_space_around_punctuation(instance['translated_text'].lower()).strip()
		mismatched = False
		for triplet in instance['translated_triplets']:
			aspect_term = add_space_around_punctuation(triplet['aspect_term'].lower()).strip()
			opinion_term = add_space_around_punctuation(triplet['opinion_term'].lower()).strip()
			if aspect_term not in translated_text and aspect_term != 'null':
				print(f"Mismatch in instance {key}: aspect_term '{aspect_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"aspect_term '{aspect_term}' not found"]
				mismatched = True
			if opinion_term not in translated_text and opinion_term != 'null':
				print(f"Mismatch in instance {key}: opinion_term '{opinion_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"opinion_term '{opinion_term}' not found"]
				mismatched = True
		if mismatched:
			mismatch_indexes.append(key)
	print(f"Total mismatches found: {len(mismatch_indexes)}")
	return mismatch_indexes, mismatch_notes
mismatch_indexes, mismatch_notes = check_mismatches_triplet_format(outputs_text)

Mismatch in instance 3550: opinion_term 'ora isa refund utawa reschedule . berhubung aku bali menyang cireboné diundur nganti tanggal 12 okt dadine kamare hangus nèk isa mbesuké ben isa reschedule' not found in sakjane , nèk saka segi pelayanan airy pancèn wis apik banget nanging . sing dadi masalah ora isa refund utawa reschedule . berhubung aku bali menyang cireboné diundur nganti tanggal 12 okt , dadine kamare hangus . nèk isa mbesuké ben isa reschedule . suwun .
Mismatch in instance 3571: opinion_term 'nomer telpon lokasi anyar aku iso nemokake alamate mergo jadwal' not found in awale angel nggoleki alamate , ning nganggo nomer telpon lokasi anyar , aku iso nemokake alamate mergo jadwal .
Mismatch in instance 3622: aspect_term 'seprei' not found in sprei karo sarung bantal ora diganti , banyu mambu , kamar ora ana cendelane .
Mismatch in instance 3666: aspect_term 'alamate' not found in alamte wae sing angel digoleki . kabeh oke .
Mismatch in instance 3698: opinion_term 'kurang api

In [ ]:
mismatch_indexes = list(set(mismatch_indexes))
len(mismatch_indexes)

17

In [ ]:
input_eng = {}
for key, instance in outputs_text.items():
	translated_text = add_space_around_punctuation(instance['translated_text'].lower()).strip()
	input_eng[key] = f'{translated_text} [A] [O] [S]'

target_eng = {}
for key, instance in outputs_text.items():
	triplet_texts = []
	for triplet in instance['translated_triplets']:
		aspect_term = add_space_around_punctuation(triplet['aspect_term'].lower()).strip()
		opinion_term = add_space_around_punctuation(triplet['opinion_term'].lower()).strip()
		sentiment = triplet['sentiment_polarity'].lower().strip()
		triplet_texts.append(f"[A] {aspect_term} [O] {opinion_term} [S] {sentiment}")
	target_eng[key] = ' [SSEP] '.join(triplet_texts)
print(len(input_eng), len(target_eng))

1000 1000


In [ ]:
# Sort input_eng and target_eng by key to match df_dataset order
input_eng = dict(sorted(input_eng.items(), key=lambda item: int(item[0])))
target_eng = dict(sorted(target_eng.items(), key=lambda item: int(item[0])))

In [ ]:
list(input_eng.values())

['pelayanane ramah banget . [A] [O] [S]',
 'eman , wifine ora apik kudu metu kamar . [A] [O] [S]',
 'tulisane twin bed , nanging sing ana kamare beda . [A] [O] [S]',
 'sakabehe apik , mung wae bakal luwih maremake menawa banyu panase iso murub 24 jam . wingi mung sedhela . [A] [O] [S]',
 'fasilitas cocok . [A] [O] [S]',
 'cedhak akses transportasi . [A] [O] [S]',
 'kamar apik cocog bujete . [A] [O] [S]',
 'pelayanane ramah , wis langganan kanggo nginep lan ngaso ing kene . [A] [O] [S]',
 'layanan kamar perlu ditingkatke . [A] [O] [S]',
 'sego gorengé enak . mbok menawa aku luwé . [A] [O] [S]',
 'ac-e kurang adem . [A] [O] [S]',
 'kamare butuh didandani . tembok kamar lan lawang-lawange . [A] [O] [S]',
 'regane pas neng kanthong , kamar wis nyukupi . [A] [O] [S]',
 'lokasi kamare terpencil . [A] [O] [S]',
 'tempate nyaman , adoh saka ramene lalu lintas kendaraan . [A] [O] [S]',
 'saben arep check in mesti ning hotel iki , amarga pelayanane apik banget . [A] [O] [S]',
 'pelayanane apik b

In [ ]:
# df_dataset = df_dataset.loc[df_dataset['sentence_id'] >= 1006, :].copy()

In [ ]:
# df_dataset.reset_index(drop=True, inplace=True)

In [ ]:
df_dataset

,sentence_id,instance_id,input,target,element_order,task_elements
0,3500,17500,pelayanan nya sangat ramah . [A] [O] [S],[A] pelayanan nya [O] sangat ramah [S] positive,aos,aos
1,3501,17505,sayang wifi tidak bagus harus keluar kamar . [...,[A] wifi [O] tidak bagus harus keluar kamar [S...,aos,aos
2,3502,17510,"tulisannya twin bed , tetapi yang ada kamarnya...",[A] kamarnya [O] beda [S] negative,aos,aos
3,3503,17515,"over all baik , hanya sja akan lebih memuaskan...",[A] over all [O] baik [S] positive [SSEP] [A] ...,aos,aos
4,3504,17520,fasilatas sesuia . [A] [O] [S],[A] fasilatas [O] sesuia [S] positive,aos,aos
...,...,...,...,...,...,...
995,4495,22475,"lumayan , harga murah banged . [A] [O] [S]",[A] harga [O] murah banged [S] positive [SSEP]...,aos,aos
996,4496,22480,buat lakilaki dan perempuan yang belum menikah...,[A] null [O] buat lakilaki dan perempuan yang ...,aos,aos
997,4497,22485,"kamar sangat nyaman dan bersih , sungguh menye...",[A] kamar [O] sangat nyaman [S] positive [SSEP...,aos,aos
998,4498,22490,"kamarnya luas , kasurnya empuk , kamar mandiny...",[A] kamarnya [O] luas [S] positive [SSEP] [A] ...,aos,aos


In [ ]:
mismatch_indexes

['3622',
 '3571',
 '3853',
 '4005',
 '4347',
 '4239',
 '3666',
 '4299',
 '4319',
 '4071',
 '3832',
 '4494',
 '3550',
 '3813',
 '4079',
 '3823',
 '3698']

In [ ]:
index_check = df_dataset['sentence_id'].to_list()
df_dataset[f'input_{lang_target}'] = list(input_eng.values())
df_dataset[f'target_{lang_target}'] = list(target_eng.values())
df_dataset['aspect_or_opinion_not_in_input'] = [str(idx) in mismatch_indexes for idx in index_check]
df_dataset['mismatch_notes'] = [mismatch_notes.get(str(idx), []) for idx in index_check]
df_dataset['mismatch_notes_format'] = df_dataset['mismatch_notes'].apply(lambda x: '\n'.join(x))

In [ ]:
df_dataset['target_format'] = df_dataset['target'].apply(lambda x: '\n'.join(x.split(' [SSEP] ')))
df_dataset[f'target_format_{lang_target}'] = df_dataset[f'target_{lang_target}'].apply(lambda x: '\n'.join(x.split(' [SSEP] ')))

In [ ]:
df_dataset

,sentence_id,instance_id,input,target,element_order,task_elements,input_jav,target_jav,aspect_or_opinion_not_in_input,mismatch_notes,mismatch_notes_format,target_format,target_format_jav
0,3500,17500,pelayanan nya sangat ramah . [A] [O] [S],[A] pelayanan nya [O] sangat ramah [S] positive,aos,aos,pelayanane ramah banget . [A] [O] [S],[A] pelayanane [O] ramah banget [S] positive,False,[],,[A] pelayanan nya [O] sangat ramah [S] positive,[A] pelayanane [O] ramah banget [S] positive
1,3501,17505,sayang wifi tidak bagus harus keluar kamar . [...,[A] wifi [O] tidak bagus harus keluar kamar [S...,aos,aos,"eman , wifine ora apik kudu metu kamar . [A] [...",[A] wifi [O] ora apik kudu metu kamar [S] nega...,False,[],,[A] wifi [O] tidak bagus harus keluar kamar [S...,[A] wifi [O] ora apik kudu metu kamar [S] nega...
2,3502,17510,"tulisannya twin bed , tetapi yang ada kamarnya...",[A] kamarnya [O] beda [S] negative,aos,aos,"tulisane twin bed , nanging sing ana kamare be...",[A] kamare [O] beda [S] negative,False,[],,[A] kamarnya [O] beda [S] negative,[A] kamare [O] beda [S] negative
3,3503,17515,"over all baik , hanya sja akan lebih memuaskan...",[A] over all [O] baik [S] positive [SSEP] [A] ...,aos,aos,"sakabehe apik , mung wae bakal luwih maremake ...",[A] sakabehe [O] apik [S] positive [SSEP] [A] ...,False,[],,[A] over all [O] baik [S] positive\n[A] air ho...,[A] sakabehe [O] apik [S] positive\n[A] banyu ...
4,3504,17520,fasilatas sesuia . [A] [O] [S],[A] fasilatas [O] sesuia [S] positive,aos,aos,fasilitas cocok . [A] [O] [S],[A] fasilitas [O] cocok [S] positive,False,[],,[A] fasilatas [O] sesuia [S] positive,[A] fasilitas [O] cocok [S] positive
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,4495,22475,"lumayan , harga murah banged . [A] [O] [S]",[A] harga [O] murah banged [S] positive [SSEP]...,aos,aos,"lumayan , regane murah banget . [A] [O] [S]",[A] rega [O] murah banget [S] positive [SSEP] ...,False,[],,[A] harga [O] murah banged [S] positive\n[A] n...,[A] rega [O] murah banget [S] positive\n[A] nu...
996,4496,22480,buat lakilaki dan perempuan yang belum menikah...,[A] null [O] buat lakilaki dan perempuan yang ...,aos,aos,kanggo lanang karo wadon sing durung rabi ya o...,[A] null [O] kanggo lanang karo wadon sing dur...,False,[],,[A] null [O] buat lakilaki dan perempuan yang ...,[A] null [O] kanggo lanang karo wadon sing dur...
997,4497,22485,"kamar sangat nyaman dan bersih , sungguh menye...",[A] kamar [O] sangat nyaman [S] positive [SSEP...,aos,aos,"kamar nyaman banget lan resik , nyenengake ten...",[A] kamar [O] nyaman banget [S] positive [SSEP...,False,[],,[A] kamar [O] sangat nyaman [S] positive\n[A] ...,[A] kamar [O] nyaman banget [S] positive\n[A] ...
998,4498,22490,"kamarnya luas , kasurnya empuk , kamar mandiny...",[A] kamarnya [O] luas [S] positive [SSEP] [A] ...,aos,aos,"kamare jembar , kasure empuk , kamar mandine y...",[A] kamare [O] jembar [S] positive [SSEP] [A] ...,False,[],,[A] kamarnya [O] luas [S] positive\n[A] kasurn...,[A] kamare [O] jembar [S] positive\n[A] kasure...


In [ ]:
df_dataset[['sentence_id', 'input', f'input_{lang_target}', 'target_format', f'target_format_{lang_target}', 'aspect_or_opinion_not_in_input', 'mismatch_notes_format']].rename({'target_format': 'target', f'target_format_{lang_target}': f'target_{lang_target}', 'mismatch_notes_format': 'mismatch_notes'}).to_csv(f'temp/translation_output/{lang_target}/{dataset_folder}/translated_dataset_{lang}_{split}.csv', index=False)

### Check mismatches after annotation

In [ ]:
def get_google_sheet(sheet_id: str, sheet_gid: str) -> pd.DataFrame:
	"""
	Downloads a specific sheet from a Google Sheet into a pandas DataFrame.

	Args:
		sheet_id: The ID of the Google Sheet.
		sheet_gid: The GID of the specific sheet to download.

	Returns:
		A pandas DataFrame containing the data from the specified sheet.
	"""
	url = f'https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={sheet_gid}'
	df = pd.read_csv(url)
	return df
google_sheet_id = '1_ZEFErp75wDNkXIvYflxv1d4nAQZrMlg_Vin4cgpyyY'  # Replace with your actual Google Sheet ID
gid_eng_train = '1363651964'  # Replace with the actual GID for the English sheet
gid_eng_test = '1183495517'
gid_eng_dev = '390220786'
gid_sunda_train = '557914217'
gid_sunda_dev = '736485835'
gid_sunda_test = '1775717233'
split = 'dev'
lang_target = 'eng'
try:
	df_translated_correction = get_google_sheet(google_sheet_id, globals()[f'gid_{lang_target}_{split}'])
	print("Successfully loaded data from the specific sheet:")
except Exception as e:
	print(f"An error occurred: {e}")
	print("Please ensure the Google Sheet is shared correctly and the IDs are correct.")

Successfully loaded data from the specific sheet:


In [ ]:
df_translated_correction

,sentence_id,input,input_eng,input_eng_corrected,target_format,target_format_eng,target_format_eng_corrected,aspect_or_opinion_not_in_input,mismatch_notes_format
0,2500,pintu geser kurang rapat . [A] [O] [S],the sliding door is not tight enough . [A] [O]...,the sliding door doesn't close properly . [A] ...,[A] pintu geser [O] kurang rapat [S] negative,[A] sliding door [O] not tight enough [S] nega...,[A] the sliding door [O] doesn't close properl...,False,NaN
1,2501,pelayanan lumayan baik . [A] [O] [S],the service is quite good . [A] [O] [S],NaN,[A] pelayanan [O] lumayan baik [S] positive,[A] service [O] quite good [S] positive,NaN,False,NaN
2,2502,air bersih untuk mck tidak ada . [A] [O] [S],clean water for sanitary facilities is not ava...,there is no clean water for the sanitary facil...,[A] air bersih [O] tidak ada [S] negative,[A] clean water [O] not available [S] negative,[A] clean water for the sanitary facilities [O...,False,NaN
3,2503,ternyata ada makanan ringan gratis . [A] [O] [S],it turns out there are free snacks . [A] [O] [S],NaN,[A] makanan ringan gratis [O] ada [S] positive,[A] free snacks [O] there are [S] positive,NaN,False,NaN
4,2504,wifi buruk suka down . [A] [O] [S],"the wifi is bad , it often goes down . [A] [O]...",NaN,[A] wifi [O] buruk [S] negative,[A] wifi [O] bad [S] negative,NaN,False,NaN
...,...,...,...,...,...,...,...,...,...
995,3495,"strategis , dekat jalan raya . [A] [O] [S]","strategic , close to the main road . [A] [O] [S]",NaN,[A] null [O] strategis [S] positive\n[A] null ...,[A] null [O] strategic [S] positive\n[A] null ...,NaN,False,NaN
996,3496,( - ) tempatnya kurang bersih . tipetipe pengi...,the place is not very clean . it ' s an old-fa...,"the downside is, the place is not very clean ....",[A] tempatnya [O] kurang bersih [S] negative\n...,[A] place [O] not very clean [S] negative\n[A]...,[A] place [O] not very clean [S] negative\n[A]...,False,NaN
997,3497,"mantapp , pelayanan prima . [A] [O] [S]","awesome , excellent service . [A] [O] [S]",NaN,[A] null [O] mantapp [S] positive\n[A] pelayan...,[A] null [O] awesome [S] positive\n[A] service...,NaN,False,NaN
998,3498,"hotel lumayan nyaman sih , tetapi dikamar saya...","the hotel was quite comfortable , but in my ro...","the hotel was quite comfortable , but in my ro...",[A] hotel [O] lumayan nyaman [S] positive\n[A]...,[A] hotel [O] quite comfortable [S] positive\n...,[A] hotel [O] quite comfortable [S] positive\n...,False,NaN


In [ ]:
df_translated_correction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   sentence_id                     1000 non-null   int64 
 1   input                           1000 non-null   object
 2   input_eng                       1000 non-null   object
 3   input_eng_corrected             534 non-null    object
 4   target_format                   1000 non-null   object
 5   target_format_eng               999 non-null    object
 6   target_format_eng_corrected     544 non-null    object
 7   aspect_or_opinion_not_in_input  1000 non-null   bool  
 8   mismatch_notes_format           40 non-null     object
dtypes: bool(1), int64(1), object(7)
memory usage: 63.6+ KB


In [ ]:
df_translated_correction[f'target_format_{lang_target}'].fillna('[A] [O] [S]', inplace=True)

/tmp/ipykernel_2375546/3226597203.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_translated_correction[f'target_format_{lang_target}'].fillna('[A] [O] [S]', inplace=True)


In [ ]:
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x.replace('’', "'") if isinstance(x, str) else x)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x.replace('’', "'") if isinstance(x, str) else x)
df_translated_correction[f'input_{lang_target}'] = df_translated_correction[f'input_{lang_target}'].apply(lambda x: x.replace('’', "'"))
df_translated_correction[f'target_format_{lang_target}'] = df_translated_correction[f'target_format_{lang_target}'].apply(lambda x: x.replace('’', "'"))

In [ ]:
list(df_translated_correction.loc[df_translated_correction['sentence_id'] == 433, f'input_{lang_target}_corrected'])

[]

In [ ]:
# Strip strings of all _corrected columns
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x.strip() if isinstance(x, str) else x)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x.strip() if isinstance(x, str) else x)

# Set to pandas nan if empty string
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].apply(lambda x: x if x != '' else np.nan)
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].apply(lambda x: x if x != '' else np.nan)

In [ ]:
# Fill _corrected columns with input_eng if null
df_translated_correction[f'input_{lang_target}_corrected'] = df_translated_correction[f'input_{lang_target}_corrected'].fillna(df_translated_correction[f'input_{lang_target}'])
df_translated_correction[f'target_format_{lang_target}_corrected'] = df_translated_correction[f'target_format_{lang_target}_corrected'].fillna(df_translated_correction[f'target_format_{lang_target}'])

In [ ]:
df_translated_correction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   sentence_id                     1000 non-null   int64 
 1   input                           1000 non-null   object
 2   input_eng                       1000 non-null   object
 3   input_eng_corrected             1000 non-null   object
 4   target_format                   1000 non-null   object
 5   target_format_eng               1000 non-null   object
 6   target_format_eng_corrected     1000 non-null   object
 7   aspect_or_opinion_not_in_input  1000 non-null   bool  
 8   mismatch_notes_format           40 non-null     object
dtypes: bool(1), int64(1), object(7)
memory usage: 63.6+ KB


In [ ]:
def check_mismatches_triplet_format_sent_id(outputs_text):
	mismatch_indexes = []
	mismatch_notes = {}
	for key, instance in outputs_text.items():
		translated_text = add_space_around_punctuation(instance['translated_text'].lower()).strip()
		mismatched = False
		for triplet in instance['translated_triplets']:
			aspect_term = add_space_around_punctuation(triplet['aspect_term'].lower()).strip()
			opinion_term = add_space_around_punctuation(triplet['opinion_term'].lower()).strip()
			if aspect_term not in translated_text and aspect_term != 'null':
				print(f"Mismatch in instance sentence_id {instance['sentence_id']} key {key}: aspect_term '{aspect_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"aspect_term '{aspect_term}' not found"]
				mismatched = True
			if opinion_term not in translated_text and opinion_term != 'null':
				print(f"Mismatch in instance sentence_id {instance['sentence_id']} key {key}: opinion_term '{opinion_term}' not found in {translated_text}")
				mismatch_notes[key] = mismatch_notes.get(key, []) + [f"opinion_term '{opinion_term}' not found"]
				mismatched = True
		if mismatched:
			mismatch_indexes.append(key)
	print(f"Total mismatches found: {len(mismatch_indexes)}")
	return mismatch_indexes, mismatch_notes

In [ ]:
outputs_text_correction = {}
for idx, row in df_translated_correction.iterrows():
	outputs_text_correction[idx] = {
		'sentence_id': row['sentence_id'],
		'translated_text': row[f'input_{lang_target}_corrected'],
		'translated_triplets': parse_absa_string(row[f'target_format_{lang_target}_corrected'].replace('\n', ' [SSEP] '))
	}
	# Change the keys of translated_triplets from A, O, S to aspect_term, opinion_term, sentiment_polarity
	for triplet in outputs_text_correction[idx]['translated_triplets']:
		triplet['aspect_term'] = triplet.pop('A', '')
		triplet['opinion_term'] = triplet.pop('O', '')
		triplet['sentiment_polarity'] = triplet.pop('S', '')

In [ ]:
len(outputs_text_correction)

1000

In [ ]:
outputs_text_correction[372]

{'sentence_id': 2872,
 'translated_text': 'my body became itchy after staying here . the room did not match the picture . [A] [O] [S]',
 'translated_triplets': [{'aspect_term': 'body',
   'opinion_term': 'itchy after staying here',
   'sentiment_polarity': 'negative'},
  {'aspect_term': 'room',
   'opinion_term': 'did not match the picture',
   'sentiment_polarity': 'negative'}]}

In [ ]:
mismatch_indexes, mismatch_notes = check_mismatches_triplet_format_sent_id(outputs_text_correction)

Mismatch in instance sentence_id 2515 key 15: opinion_term 'wait a long time' not found in when we checked out , there was no staff available , so we had to wait for a long time . [a] [o] [s]
Mismatch in instance sentence_id 2566 key 66: aspect_term 'the partition wall' not found in the lobby space is too far in . if possible , the partitition wall could be opened so that the outside view can be seen . [a] [o] [s]
Mismatch in instance sentence_id 2584 key 84: opinion_term 'strategic' not found in the location is convenient , near malang city square , with many culinary places . [a] [o] [s]
Mismatch in instance sentence_id 2614 key 114: opinion_term 'the best' not found in please arrange the maintainance for the elevator , but everything else was excellent ! [a] [o] [s]
Mismatch in instance sentence_id 2623 key 123: opinion_term 'soomewhat disappointing' not found in the bathroom was a bit dirty , and the food was also somewhat disappointing . i had breakfast at 8 am , but all of the fo

In [ ]:
df_translated_correction['aspect_or_opinion_not_in_input'] = [idx in mismatch_indexes for idx in df_translated_correction.index]
df_translated_correction['mismatch_notes'] = [mismatch_notes.get(idx, []) for idx in df_translated_correction.index]
df_translated_correction['mismatch_notes_format'] = df_translated_correction['mismatch_notes'].apply(lambda x: '\n'.join(x))

In [ ]:
os.makedirs(f'temp/translation_output/{lang_target}/{dataset_folder}', exist_ok=True)
df_translated_correction[['aspect_or_opinion_not_in_input', 'mismatch_notes_format']].to_csv(f'temp/translation_output/{lang_target}/{dataset_folder}/translated_dataset_{lang}_{split}_correction_mismatches.csv', index=False)